## *SOME PREPARATION*

In [28]:
# import pandas as pd

In [2]:
# df = pd.read_csv('/content/drive/MyDrive/preprocessed csv/data.csv')

In [3]:
# df.head()

In [4]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

# print("Path to dataset files:", path)
# import os
# print(os.listdir(path))

In [5]:
# df2 = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [6]:
# df2.head()

In [7]:
# print(df2.shape, df.shape)

In [8]:
# dfNew = pd.DataFrame({'review': df['reviewLemmatized'], 'sentiment': df2['sentiment']})

In [9]:
# dfNew.head()

In [10]:
# dfNew.to_csv('/content/drive/MyDrive/preprocessed csv/dataFinal.csv', index=False)

In [11]:
# dfA = pd.read_csv('/content/drive/MyDrive/preprocessed csv/dataFinal.csv')

In [12]:
# dfA.head()

# *Few checks*

In [1]:
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
df = pd.read_csv('/content/drive/MyDrive/preprocessed csv/dataFinal.csv')
# df = df[:20000] # couldn't use all cause the RAM was exhausted during BOW creation
# df = df[:12000] # couldn't use all cause the RAM was exhausted  This is for GaussianNB
# df = df[:15000]
df = df[:10000]

In [5]:
print(df.isnull().sum())
print(df.duplicated().sum())
df.drop_duplicates(inplace=True)
print(df.duplicated().sum())

review       0
sentiment    0
dtype: int64
17
0


In [6]:
df.shape[0]

9983

In [7]:
df.head()

,review,sentiment
0,one reviewer mention watch 1 oz episode you ll...,positive
1,wonderful little production film technique una...,positive
2,think wonderful way spend time hot summer week...,positive
3,basically there s family little boy jake think...,negative
4,petter matteis love time money visually stunni...,positive


In [8]:
df['sentiment'].value_counts()

,count
sentiment,
positive,5023
negative,4960


In [9]:
# CREATING X AND Y variable to start actions
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [10]:
X.head()

,review
0,one reviewer mention watch 1 oz episode you ll...
1,wonderful little production film technique una...
2,think wonderful way spend time hot summer week...
3,basically there s family little boy jake think...
4,petter matteis love time money visually stunni...


In [11]:
y.head()

,sentiment
0,positive
1,positive
2,positive
3,negative
4,positive


In [12]:
# LABEL ENCODING THE sentiment column
from sklearn.preprocessing import LabelEncoder
leObj = LabelEncoder()
y = leObj.fit_transform(y)

In [13]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [14]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [15]:
X_train

,review
6970,know story sweeney todd likely thanks tim burt...
5261,jane russell underrate comedienne singer see s...
6810,upon time director name james bring u wonderfu...
6558,movie disaster within disaster film full great...
7369,absolutely nothing movie funny interesting rel...
...,...
9240,event define era wrestling entertainment belie...
4862,many horrible spoof movie sadly breath fresh a...
3264,want like one situation rich set unusual inter...
9862,enjoy movie have nt see andy griffith age felt...


# *DATA REPRESENTATION*

## *BOW*

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
# cvObj = CountVectorizer()
cvObj = CountVectorizer(max_features=10000)

In [27]:
X_train_bow = cvObj.fit_transform(X_train['review']).toarray()
X_test_bow = cvObj.transform(X_test['review']).toarray()

### *MODEL TRAINING*

#####  *Naive Bayes* *italicized text*

In [ ]:
import time
start = time.time()
from sklearn.naive_bayes import MultinomialNB
nbObj = MultinomialNB()
nbObj.fit(X_train_bow, y_train)
end = time.time()
MultinomialNB_TimeTaken = end-start
print(MultinomialNB_TimeTaken)

65.91242551803589


In [ ]:
y_pred = nbObj.predict(X_test_bow)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
MultinomialNB_accuracy_score = accuracy_score(y_test, y_pred)
MultinomialNB_confusion_matrix = confusion_matrix(y_test, y_pred)
print(MultinomialNB_accuracy_score)
print(MultinomialNB_confusion_matrix)

0.8486825595984944
[[1778  274]
 [ 329 1604]]


In [ ]:
# CREATING A FOLDER TO SAVE THE MODELS and the achieved metrics
import joblib
import os
modelfolderPath = "/content/drive/MyDrive/project_models/models"
logsfolderPath = "/content/drive/MyDrive/project_models/logs"

if not os.path.exists(modelfolderPath):
  os.makedirs(modelfolderPath)
if not os.path.exists(logsfolderPath):
  os.makedirs(logsfolderPath)

# Save the nbObj
model_path = os.path.join(modelfolderPath, "MultinomialNB.pkl")
joblib.dump(nbObj, model_path)

print(f"Model saved to {model_path}")

# Save log
log_file_path = os.path.join(logsfolderPath, "log_MultinomialNB.txt")

with open(log_file_path, "a") as f:
    f.write("========== Training Run ==========\n")
    f.write(f"Model: MultinomialNB\n")
    f.write(f"Time Taken: {MultinomialNB_TimeTaken} seconds\n")
    f.write(f"Accuracy: {MultinomialNB_accuracy_score:.4f}\n")
    f.write(f"Confusion Matrix:\n{MultinomialNB_confusion_matrix}\n\n")
    f.write(f"Dataset size:\n{df.shape[0]}\n\n")

print("✅ Model and logs saved successfully!")

Model saved to /content/drive/MyDrive/project_models/models/MultinomialNB.pkl
✅ Model and logs saved successfully!


In [ ]:
import time
start = time.time()
from sklearn.naive_bayes import GaussianNB
gnbObj = GaussianNB()
gnbObj.fit(X_train_bow, y_train)
end = time.time()
GaussianNB_TimeTaken = end-start
print(GaussianNB_TimeTaken)

11.677737951278687


In [ ]:
y_pred = gnbObj.predict(X_test_bow)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
GaussianNB_accuracy_score = accuracy_score(y_test, y_pred)
GaussianNB_confusion_matrix = confusion_matrix(y_test, y_pred)
print(GaussianNB_accuracy_score)
print(GaussianNB_confusion_matrix)

0.6494156928213689
[[906 298]
 [542 650]]


In [ ]:
# CREATING A FOLDER TO SAVE THE MODELS and the achieved metrics
import joblib
import os
modelfolderPath = "/content/drive/MyDrive/project_models/models"
logsfolderPath = "/content/drive/MyDrive/project_models/logs"

if not os.path.exists(modelfolderPath):
  os.makedirs(modelfolderPath)
if not os.path.exists(logsfolderPath):
  os.makedirs(logsfolderPath)

# Save the nbObj
model_path = os.path.join(modelfolderPath, "GaussianNB.pkl")
joblib.dump(gnbObj, model_path)

print(f"Model saved to {model_path}")

# Save log
log_file_path = os.path.join(logsfolderPath, "log_GaussianNB.txt")

with open(log_file_path, "a") as f:
    f.write("========== Training Run ==========\n")
    f.write(f"Model: GaussianNB\n")
    f.write(f"Time Taken: {GaussianNB_TimeTaken} seconds\n")
    f.write(f"Accuracy: {GaussianNB_accuracy_score:.4f}\n")
    f.write(f"Confusion Matrix:\n{GaussianNB_confusion_matrix}\n\n")
    f.write(f"Dataset size:\n{df.shape[0]}\n\n")


print("✅ Model and logs saved successfully!")

Model saved to /content/drive/MyDrive/project_models/models/GaussianNB.pkl
✅ Model and logs saved successfully!


##### *Random Forest*

In [30]:
import time
start = time.time()
from sklearn.ensemble import RandomForestClassifier
rfcObj = RandomForestClassifier()
rfcObj.fit(X_train_bow, y_train)
end = time.time()
RF_TimeTaken = end-start
print(RF_TimeTaken)

48.704413414001465


In [33]:
y_pred = rfcObj.predict(X_test_bow)

In [34]:
from sklearn.metrics import accuracy_score, confusion_matrix
RF_accuracy_score = accuracy_score(y_test, y_pred)
RF_confusion_matrix = confusion_matrix(y_test, y_pred)
print(RF_accuracy_score)
print(RF_confusion_matrix)

0.8411543287327478
[[1719  333]
 [ 300 1633]]


In [36]:
# CREATING A FOLDER TO SAVE THE MODELS and the achieved metrics
import joblib
import os
modelfolderPath = "/content/drive/MyDrive/project_models/models"
logsfolderPath = "/content/drive/MyDrive/project_models/logs"

if not os.path.exists(modelfolderPath):
  os.makedirs(modelfolderPath)
if not os.path.exists(logsfolderPath):
  os.makedirs(logsfolderPath)

# Save the nbObj
model_path = os.path.join(modelfolderPath, "RF.pkl")
joblib.dump(rfcObj, model_path)

print(f"Model saved to {model_path}")

# Save log
log_file_path = os.path.join(logsfolderPath, "log_RF.txt")

with open(log_file_path, "a") as f:
    f.write("========== Training Run ==========\n")
    f.write(f"Model: RF\n")
    f.write(f"Time Taken: {RF_TimeTaken} seconds\n")
    f.write(f"Accuracy: {RF_accuracy_score:.4f}\n")
    f.write(f"Confusion Matrix:\n{RF_confusion_matrix}\n\n")
    f.write(f"Dataset size:\n{df.shape[0]}\n\n")


print("✅ Model and logs saved successfully!")

Model saved to /content/drive/MyDrive/project_models/models/RF.pkl
✅ Model and logs saved successfully!


## *Tfidf*

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidfObj = TfidfVectorizer()
X_train_tfidf = (tfidfObj.fit_transform(X_train['review'])).toarray()
X_test_tfidf = (tfidfObj.transform(X_test['review'])).toarray()

In [27]:
X_train_tfidf.shape

(7986, 64985)

### *MODEL TRAINING*

#####  *Naive Bayes*

In [23]:
import time
start = time.time()
from sklearn.naive_bayes import MultinomialNB
nb_tfidf_Obj = MultinomialNB()
nb_tfidf_Obj.fit(X_train_tfidf, y_train)
end = time.time()
MultinomialNB_tfidf_TimeTaken = end-start
print(MultinomialNB_tfidf_TimeTaken)

2.159390926361084


In [24]:
y_pred = nb_tfidf_Obj.predict(X_test_tfidf)

In [25]:
from sklearn.metrics import accuracy_score, confusion_matrix
MultinomialNB_tfidf_accuracy_score = accuracy_score(y_test, y_pred)
MultinomialNB_tfidf_confusion_matrix = confusion_matrix(y_test, y_pred)
print(MultinomialNB_tfidf_accuracy_score)
print(MultinomialNB_tfidf_confusion_matrix)

0.8349482124958236
[[1331  118]
 [ 376 1168]]


In [27]:
# CREATING A FOLDER TO SAVE THE MODELS and the achieved metrics
import joblib
import os
modelfolderPath = "/content/drive/MyDrive/project_models/models"
logsfolderPath = "/content/drive/MyDrive/project_models/logs"

if not os.path.exists(modelfolderPath):
  os.makedirs(modelfolderPath)
if not os.path.exists(logsfolderPath):
  os.makedirs(logsfolderPath)

# Save the nbObj
model_path = os.path.join(modelfolderPath, "MultinomialNB_tfidf.pkl")
joblib.dump(nb_tfidf_Obj, model_path)

print(f"Model saved to {model_path}")

# Save log
log_file_path = os.path.join(logsfolderPath, "log_MultinomialNB_tfidf.txt")

with open(log_file_path, "a") as f:
    f.write("========== Training Run ==========\n")
    f.write(f"Model: MultinomialNB_tfidf\n")
    f.write(f"Time Taken: {MultinomialNB_tfidf_TimeTaken} seconds\n")
    f.write(f"Accuracy: {MultinomialNB_tfidf_accuracy_score:.4f}\n")
    f.write(f"Confusion Matrix:\n{MultinomialNB_tfidf_confusion_matrix}\n\n")
    f.write(f"Dataset size:\n{df.shape[0]}\n\n")

print("✅ Model and logs saved successfully!")

Model saved to /content/drive/MyDrive/project_models/models/MultinomialNB_tfidf.pkl
✅ Model and logs saved successfully!


In [18]:
import time
start = time.time()
from sklearn.naive_bayes import GaussianNB
gnb_tfidf_Obj = GaussianNB()
gnb_tfidf_Obj.fit(X_train_tfidf, y_train)
end = time.time()
GaussianNB_tfidf_TimeTaken = end-start
print(GaussianNB_tfidf_TimeTaken)

7.5979578495025635


In [20]:
y_pred = gnb_tfidf_Obj.predict(X_test_tfidf)

In [21]:
from sklearn.metrics import accuracy_score, confusion_matrix
GaussianNB_tfidf_accuracy_score = accuracy_score(y_test, y_pred)
GaussianNB_tfidf_confusion_matrix = confusion_matrix(y_test, y_pred)
print(GaussianNB_tfidf_accuracy_score)
print(GaussianNB_tfidf_confusion_matrix)

0.6374561842764146
[[656 325]
 [399 617]]


In [22]:
# CREATING A FOLDER TO SAVE THE MODELS and the achieved metrics
import joblib
import os
modelfolderPath = "/content/drive/MyDrive/project_models/models"
logsfolderPath = "/content/drive/MyDrive/project_models/logs"

if not os.path.exists(modelfolderPath):
  os.makedirs(modelfolderPath)
if not os.path.exists(logsfolderPath):
  os.makedirs(logsfolderPath)

# Save the nbObj
model_path = os.path.join(modelfolderPath, "GaussianNB_tfidf.pkl")
joblib.dump(gnb_tfidf_Obj, model_path)

print(f"Model saved to {model_path}")

# Save log
log_file_path = os.path.join(logsfolderPath, "log_GaussianNB_tfidf.txt")

with open(log_file_path, "a") as f:
    f.write("========== Training Run ==========\n")
    f.write(f"Model: GaussianNB_tfidf\n")
    f.write(f"Time Taken: {GaussianNB_tfidf_TimeTaken} seconds\n")
    f.write(f"Accuracy: {GaussianNB_tfidf_accuracy_score:.4f}\n")
    f.write(f"Confusion Matrix:\n{GaussianNB_tfidf_confusion_matrix}\n\n")
    f.write(f"Dataset size:\n{df.shape[0]}\n\n")


print("✅ Model and logs saved successfully!")

Model saved to /content/drive/MyDrive/project_models/models/GaussianNB_tfidf.pkl
✅ Model and logs saved successfully!


##### *Random Forest*

In [23]:
import time
start = time.time()
from sklearn.ensemble import RandomForestClassifier
rfc_tfidf_Obj = RandomForestClassifier()
rfc_tfidf_Obj.fit(X_train_tfidf, y_train)
end = time.time()
RF__tfidf_TimeTaken = end-start
print(RF__tfidf_TimeTaken)

85.83629655838013


In [24]:
y_pred = rfc_tfidf_Obj.predict(X_test_tfidf)

In [25]:
from sklearn.metrics import accuracy_score, confusion_matrix
RF__tfidf_accuracy_score = accuracy_score(y_test, y_pred)
RF__tfidf_confusion_matrix = confusion_matrix(y_test, y_pred)
print(RF__tfidf_accuracy_score)
print(RF__tfidf_confusion_matrix)

0.8492739108662994
[[849 132]
 [169 847]]


In [26]:
# CREATING A FOLDER TO SAVE THE MODELS and the achieved metrics
import joblib
import os
modelfolderPath = "/content/drive/MyDrive/project_models/models"
logsfolderPath = "/content/drive/MyDrive/project_models/logs"

if not os.path.exists(modelfolderPath):
  os.makedirs(modelfolderPath)
if not os.path.exists(logsfolderPath):
  os.makedirs(logsfolderPath)

# Save the nbObj
model_path = os.path.join(modelfolderPath, "RF_tfidf.pkl")
joblib.dump(rfc_tfidf_Obj, model_path)

print(f"Model saved to {model_path}")

# Save log
log_file_path = os.path.join(logsfolderPath, "log_RF_tfidf.txt")

with open(log_file_path, "a") as f:
    f.write("========== Training Run ==========\n")
    f.write(f"Model: RF\n")
    f.write(f"Time Taken: {RF__tfidf_TimeTaken} seconds\n")
    f.write(f"Accuracy: {RF__tfidf_accuracy_score:.4f}\n")
    f.write(f"Confusion Matrix:\n{RF__tfidf_confusion_matrix}\n\n")
    f.write(f"Dataset size:\n{df.shape[0]}\n\n")


print("✅ Model and logs saved successfully!")

Model saved to /content/drive/MyDrive/project_models/models/RF_tfidf.pkl
✅ Model and logs saved successfully!
